# 🎓 Fundamentos de Sistemas de Recuperação da Informação (IR) e TF-IDF

**Professor Doutor em Engenharia de Inteligência Artificial**

--- 

## 🎯 Objetivos de Revisão

Esta série de aulas cobre os fundamentos da conversão de texto em formatos que os modelos de IA podem processar, culminando na implementação prática do **Vector Space Model (VSM)** utilizando a métrica **TF-IDF (Term Frequency-Inverse Document Frequency)** para ranqueamento de documentos.

### 🗺️ Roteiro:
1. **Tokenização (Pré-processamento):** Divisão de texto em unidades gerenciáveis.
2. **Vector Space Model (VSM):** Representação matemática de texto (vetores e dimensões).
3. **TF-IDF:** Ponderação estatística para medir a importância de um termo.
4. **Ranqueamento:** Cálculo da Similaridade do Cosseno para ordenação de resultados.

# 1. Tokenização: A Base do Processamento de Texto

A tokenização é o processo de dividir um texto em unidades menores chamadas **tokens**. É o primeiro passo para que qualquer algoritmo de NLP possa processar a linguagem humana.

Tipos de Tokenização revisados:
*   **Word Tokenization:** Divide o texto em palavras individuais.
*   **Sentence Tokenization:** Divide o texto em frases completas.
*   **Subword Tokenization (LLMs):** Divide palavras em unidades menores para eficiência (como visto na ferramenta OpenAI Tokenizer).

In [ ]:
# Instalação e Importação de Bibliotecas Essenciais

import nltk
from nltk.corpus import stopwords
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import re

# O NLTK requer o download de pacotes de dados específicos (corpora e tokenizers)
try:
    # Pacote 'punkt' é necessário para sent_tokenize (tokenização de sentença)
    nltk.data.find('tokenizers/punkt')
    # Pacote 'stopwords' é necessário para o NLTK saber quais palavras remover
    nltk.data.find('corpora/stopwords')
except nltk.downloader.DownloadError:
    print("Baixando pacotes NLTK: punkt e stopwords...")
    nltk.download('punkt')
    nltk.download('stopwords')
    print("Downloads concluídos!")

print("Bibliotecas NLTK e Scikit-learn importadas com sucesso!")

In [ ]:
# Exemplo de Tokenização (Word e Sentence Tokenization)

text = "Machine learning é um campo da inteligência artificial que permite que computadores aprendam padrões a partir de dados, sem serem programados explicitamente para cada tarefa."

# 1. Word Tokenization (Tokens por palavra)
word_tokens = nltk.word_tokenize(text)
print("\n--- Word Tokens ---")
print(word_tokens)

# 2. Sentence Tokenization (Tokens por sentença)
sentence_tokens = nltk.sent_tokenize(text)
print("\n--- Sentence Tokens ---")
print(sentence_tokens)

# 2. Vector Space Model (VSM) e TF-IDF

O VSM permite que o texto seja tratado matematicamente. Ele representa documentos e consultas como **vetores** em um espaço multidimensional. 

**Regras Chave do VSM:**
1.  **Dimensões:** Cada termo único no corpus (após tokenização e pré-processamento) torna-se uma dimensão no espaço vetorial.
2.  **Peso:** O peso de cada termo no vetor é calculado usando o TF-IDF.
3.  **Similaridade:** A relevância é medida pela **Similaridade do Cosseno** (ângulo entre o vetor da query e os vetores dos documentos). 

### Detalhe: Cálculo do TF-IDF

O TF-IDF é o produto de duas métricas:

1. **Term Frequency (TF) - Importância Local:** Mede a frequência do termo no documento.
$$TF(t, d) = \frac{\text{Número de vezes que o termo } t \text{ aparece no documento } d}{\text{Número total de termos no documento } d}$$

2. **Inverse Document Frequency (IDF) - Raridade Global:** Pondera a importância do termo em relação a todo o corpus (penalizando palavras muito comuns, como *stop words*).
$$IDF(t) = \log \left( \frac{\text{Número total de documentos (N)}}{\text{Número de documentos que contêm o termo } t} \right)$$

3. **TF-IDF Final:**
$$TF-IDF(t, d) = TF(t, d) \times IDF(t)$$

# 3. Implementação Prática com Scikit-learn (TF-IDF Ranqueamento)

Para simular um sistema de recuperação da informação, utilizaremos um corpus de 11 documentos e uma função de pré-processamento robusta que garante a normalização do texto.

In [ ]:
# Corpus de Documentos (Base de Conhecimento)
documents = [
    "Machine learning é um campo da inteligência artificial que permite que computadores aprendam padrões a partir de dados.",
    "O aprendizado de máquina dá aos sistemas a capacidade de melhorar seu desempenho sem serem explicitamente programados.",
    "Em vez de seguir apenas regras fixas, o machine learning descobre relações escondidas nos dados.",
    "Esse campo combina estatística, algoritmos e poder computacional para extrair conhecimento.",
    "O objetivo é criar modelos capazes de generalizar além dos exemplos vistos no treinamento.",
    "Aplicações de machine learning vão desde recomendações de filmes até diagnósticos médicos.",
    "Os algoritmos de aprendizado de máquina transformam dados brutos em previsões úteis.",
    "Diferente de um software tradicional, o ML adapta-se conforme novos dados chegam.",
    "O aprendizado pode ser supervisionado, não supervisionado ou por reforço, dependendo do tipo de problema.",
    "Na prática, machine learning é o motor que impulsiona muitos avanços em visão computacional e processamento de linguagem natural.",
    "Mais do que encontrar padrões, o machine learning ajuda a tomar decisões baseadas em evidências."
]

# Função de Pré-processamento (Limpeza e Normalização)
def preprocess(text):
    """Tokeniza, converte para minúsculas e remove caracteres não alfanuméricos."""
    text_lower = text.lower()
    # Usamos o word_tokenize do NLTK para uma tokenização mais sofisticada (lidando com pontuações/contractions)
    tokens = nltk.word_tokenize(text_lower)
    
    # Filtro: mantém apenas tokens alfanuméricos (remover pontuações, etc.)
    return [word for word in tokens if word.isalnum()]

# Aplica o pré-processamento em todos os documentos e os une novamente em strings
preprocessed_docs = []
for doc in documents:
    tokens = preprocess(doc)
    # Junta os tokens com um espaço simples (o TfidfVectorizer espera strings)
    preprocessed_docs.append(" ".join(tokens))

print(f"Primeiro documento processado: '{preprocessed_docs[0]}'" )

In [ ]:
# 4. Configuração e Treinamento do TF-IDF Vectorizer
print("\n--- 4. Configurando e Treinando o TfidfVectorizer ---")
stop_words_pt = stopwords.words('portuguese')

vectorizer = TfidfVectorizer(
    stop_words=stop_words_pt,
    # A função lambda trata a entrada como já tokenizada por espaços (pré-processamento já feito)
    tokenizer=lambda x: x.split(), 
    token_pattern=None # Desabilita o token_pattern padrão
)

# 5. Fit e Transformação do Corpus
# tfidf_matrix será uma matriz esparsa (documentos x termos)
tfidf_matrix = vectorizer.fit_transform(preprocessed_docs)

num_docs, num_terms = tfidf_matrix.shape
print(f"Shape da Matriz TF-IDF: {num_docs} documentos, {num_terms} termos (dimensões)")
print(f"Total de elementos armazenados na matriz esparsa: {tfidf_matrix.nnz}")

# Exibição (apenas para fins didáticos, mostrando as primeiras 3 linhas e 5 colunas)
print("\n--- Matriz TF-IDF (Primeiras 3 linhas) ---")
print(tfidf_matrix.toarray()[:3, :5].round(4))

In [ ]:
# 6. Função de Busca e Ranqueamento (Lógica do VSM)
def search_tfidf(query, vectorizer, tfidf_matrix, top_n=5):
    """ Executa a busca por similaridade do cosseno e ranqueia os documentos. """
    # Transforma a Query (usando o vocabulário e IDF do vectorizer treinado)
    processed_query = [" ".join(preprocess(query))]
    query_vector = vectorizer.transform(processed_query)
    
    # Calcula a Similaridade do Cosseno entre a Query e todos os Documentos
    similarities_matrix = cosine_similarity(tfidf_matrix, query_vector)
    
    # Converte para um array 1D (facilita o enumerate)
    similarities = similarities_matrix.flatten()
    
    # Associa o score de similaridade ao índice original do documento
    indexed_similarities = list(enumerate(similarities))
    
    # Ordena pelo score (índice 1 da tupla), em ordem decrescente (reverse=True)
    results = sorted(indexed_similarities, key=lambda x: x[1], reverse=True)
    
    # Retorna os Top N resultados
    return results[:top_n]

# 7. Execução e Exibição dos Resultados Ranqueados
query_text = "machine learning"
top_n_results = 5

print(f"\n--- 7. Resultados Ranqueados para a Query: '{query_text}' ---")

top_results = search_tfidf(query_text, vectorizer, tfidf_matrix, top_n=top_n_results)

print("\n[Documentos Ranqueados por Similaridade do Cosseno]:")
for rank, (doc_index, score) in enumerate(top_results):
    # Recupera o documento original a partir do índice ranqueado
    documento_original = documents[doc_index]
    
    print(f"\n--- Rank {rank + 1} | Documento {doc_index} (Score: {score:.6f}) ---")
    print(documento_original)
    
print("\nExecução concluída com sucesso!")